# 🎨 AI Text-to-Image Generator with Stable Diffusion

A modular, optimized, and feature-complete Jupyter Notebook for generating high-resolution images from text prompts using **Stable Diffusion v1.5** and Hugging Face `diffusers`.

### Key Features:
- ⚡ **Fast DPM++ 2M Karras Scheduler**: High quality in 20–30 steps.
- 🛡️ **Clean & Secure**: Device auto-detection (CUDA / Apple MPS / CPU), safe token handling.
- 🎯 **Full Parameter Control**: Negative prompts, Classifier-Free Guidance (CFG), custom resolutions, seeds, and style presets.
- 💾 **Auto-Save & Metadata**: Automatically exports generated images to `outputs/` with prompt information.

## 1. Install & Upgrade Dependencies

In [ ]:
!pip install diffusers transformers accelerate safetensors gradio matplotlib pillow --upgrade -q
print("✅ Dependencies installed successfully.")

## 2. Imports & Hardware Auto-Detection

In [ ]:
import os
import time
import json
from datetime import datetime
from PIL import Image
import matplotlib.pyplot as plt
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

# Auto-detect compute device & appropriate precision
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
    gpu_name = torch.cuda.get_device_name(0)
    print(f"🚀 Running on GPU: {gpu_name} (FP16 enabled)")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
    print("🍎 Running on Apple Silicon MPS (FP16 enabled)")
else:
    device = "cpu"
    dtype = torch.float32
    print("💻 Running on CPU (FP32 precision)")

## 3. Load Model Pipeline & Fast Scheduler
We load `runwayml/stable-diffusion-v1-5` with `safetensors` and configure the **DPM-Solver++ 2M Karras** multi-step scheduler for fast and crisp sampling.

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"
print(f"📦 Loading model: {model_id}...")

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=dtype,
    use_safetensors=True
)

# Attach high-performance DPM-Solver scheduler
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    use_karras_sigmas=True,
    algorithm_type="dpmsolver++"
)

# Enable memory optimizations
pipe.enable_attention_slicing()
pipe = pipe.to(device)

# Ensure outputs directory exists
os.makedirs("outputs", exist_ok=True)
print("✨ Model loaded and ready!")

## 4. Parameterized Generation Function & Style Presets
Below is a clean generation helper with support for style presets, negative prompts, seeds, guidance scale, and automatic file saving.

In [ ]:
STYLE_PRESETS = {
    "None": "",
    "Photorealistic": "photorealistic, 8k resolution, highly detailed, professional photography, natural lighting, sharp focus",
    "Cinematic": "cinematic still, dramatic lighting, 35mm photograph, film grain, depth of field, blockbuster movie aesthetic",
    "Anime": "anime artwork, vivid colors, Studio Ghibli style, clean linework, highly detailed illustration",
    "Digital Art": "digital concept art, trending on ArtStation, dynamic composition, vibrant palette, fantasy illustration",
    "3D Render": "octane 3D render, Unreal Engine 5, raytracing, volumetric lighting, photorealistic textures, 8k",
    "Cyberpunk": "cyberpunk style, neon lights, futuristic cityscape, rainy reflections, high tech, atmospheric glow",
    "Oil Painting": "oil on canvas, visible brushstrokes, textured, classical fine art masterpiece, rich pigments"
}

DEFAULT_NEGATIVE = (
    "blurry, low quality, distorted, deformed, bad anatomy, bad hands, "
    "missing fingers, extra limbs, duplicate, ugly, text, watermark, signature, grainy"
)

def generate_image(
    prompt: str,
    negative_prompt: str = DEFAULT_NEGATIVE,
    style_preset: str = "None",
    num_inference_steps: int = 30,
    guidance_scale: float = 7.5,
    width: int = 512,
    height: int = 512,
    seed: int = None,
    save_path: str = None
):
    """Generate an image with full parameter controls and display it."""
    # Apply style preset
    style_suffix = STYLE_PRESETS.get(style_preset, "")
    final_prompt = f"{prompt.strip()}, {style_suffix}" if style_suffix else prompt.strip()
    
    if seed is None:
        seed = torch.randint(0, 2**32 - 1, (1,)).item()
    
    generator = torch.Generator(device=device).manual_seed(seed)
    
    print(f"🎨 Generating with seed={seed}, steps={num_inference_steps}, CFG={guidance_scale}...")
    start_time = time.time()
    
    with torch.inference_mode():
        result = pipe(
            prompt=final_prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            width=width,
            height=height,
            generator=generator,
        )
    
    elapsed = time.time() - start_time
    image = result.images[0]
    
    # Save image
    if save_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = f"outputs/img_{timestamp}_s{seed}.png"
    
    image.save(save_path)
    print(f"✅ Done in {elapsed:.2f}s! Saved to: {save_path}")
    
    # Display image
    plt.figure(figsize=(7, 7))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"{prompt[:50]}... (Seed: {seed})", fontsize=10)
    plt.show()
    
    return image, save_path

## 5. Generate a Custom Image
Modify the prompt and parameters below to generate your image.

In [ ]:
my_prompt = "A serene young woman sitting peacefully on a wooden bench in a vibrant sunlit garden with blooming flowers and butterflies"

image, path = generate_image(
    prompt=my_prompt,
    style_preset="Photorealistic",
    num_inference_steps=30,
    guidance_scale=7.5,
    seed=42
)

## 6. Style Comparison Matrix
Compare how the same base prompt renders across multiple artistic styles.

In [ ]:
test_prompt = "A mystical dragon perched on a crystal mountain"
styles_to_test = ["Photorealistic", "Anime", "Digital Art", "Cyberpunk"]
fixed_seed = 100

fig, axes = plt.subplots(1, len(styles_to_test), figsize=(18, 5))

for idx, style in enumerate(styles_to_test):
    print(f"Generating style: {style}...")
    img, _ = generate_image(
        prompt=test_prompt,
        style_preset=style,
        seed=fixed_seed,
        num_inference_steps=25,
    )
    axes[idx].imshow(img)
    axes[idx].axis("off")
    axes[idx].set_title(style, fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

## 7. Interactive In-Notebook Web App (Gradio)
Launch an interactive UI directly inside your notebook.

In [ ]:
import gradio as gr

def interactive_generate(prompt, style, steps, cfg, seed):
    seed_val = int(seed) if seed > 0 else None
    img, path = generate_image(
        prompt=prompt,
        style_preset=style,
        num_inference_steps=int(steps),
        guidance_scale=float(cfg),
        seed=seed_val
    )
    return img

demo = gr.Interface(
    fn=interactive_generate,
    inputs=[
        gr.Textbox(label="Prompt", value="A majestic castle on a cloud at sunset, fantasy art"),
        gr.Dropdown(label="Style", choices=list(STYLE_PRESETS.keys()), value="Digital Art"),
        gr.Slider(label="Steps", minimum=15, maximum=50, value=30, step=1),
        gr.Slider(label="CFG Scale", minimum=1.0, maximum=15.0, value=7.5, step=0.5),
        gr.Number(label="Seed (-1 for random)", value=-1)
    ],
    outputs=gr.Image(label="Generated Image"),
    title="AI Text-to-Image Studio"
)

# To launch inline in Colab or Jupyter:
demo.launch(inline=True, share=False)